In [1]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load(p):
    df = pd.read_csv(p, header=None, names=["ts","container","image","status"])
    df["func"] = df["image"].str.split(":").str[-1].str.split("-v").str[0]
    df["status"] = df["status"].str.strip().str.lower()
    df["t"] = (df["ts"] - df["ts"].min())/1000.0
    return df

def metrics(p):
    df = load(p)
    sec = df.groupby("status").size()
    run = sec.get("running",0); pau = sec.get("paused",0); rem = sec.get("removing",0)
    total = run+pau+rem
    util = run/total if total else 0
    g = df.groupby("container")
    lifespan = (g["t"].max()-g["t"].min()).mean()
    return {"util": util, "idle": (pau+rem)/total, "lifespan": lifespan,
            "run_s": run, "pau_s": pau}

baseline_docker_path = "../results/exp_20260602_170520/docker_log.csv"
proposed_docker_path = "../results/exp_20260602_154435/docker_log.csv"
b = metrics(baseline_docker_path)
p = metrics(proposed_docker_path)
print("baseline:", b)
print("proposed:", p)

# Middleware-style figure: two panels, clean, publication colors
plt.rcParams.update({"font.size": 11})
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

systems = ["OpenWhisk", "NMIG"]
x = np.arange(2)

# Panel 1: active vs idle container-time (stacked, %)
active = np.array([b["util"], p["util"]])*100
idle   = np.array([b["idle"], p["idle"]])*100
ax = axes[0]
ax.bar(x, active, 0.55, label="Active (running)", color="#2a9d8f")
ax.bar(x, idle, 0.55, bottom=active, label="Idle (paused)", color="#e76f51")
for i in range(2):
    ax.text(i, active[i]/2, f"{active[i]:.1f}%", ha="center", va="center",
            color="white", fontweight="bold", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel("Container lifetime (%)")
ax.set_title("(a) Container utilization")
ax.legend(fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.42), ncol=2)
ax.set_ylim(0, 100)

# Panel 2: mean container lifespan
ax = axes[1]
life = [b["lifespan"], p["lifespan"]]
bars = ax.bar(x, life, 0.55, color=["#264653", "#2a9d8f"])
for i, v in enumerate(life):
    ax.text(i, v, f"{v:.0f}s", ha="center", va="bottom", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel("Mean container lifespan (s)")
ax.set_title("(b) Idle retention (lifespan)")

fig.tight_layout()
fig.savefig("container_outputs/container_utilization.pdf", bbox_inches="tight")
fig.savefig("container_outputs/container_utilization.png", dpi=200, bbox_inches="tight")
print("saved figure")

baseline: {'util': 0.09963558413719185, 'idle': 0.9003644158628081, 'lifespan': 1736.142857142857, 'run_s': 2324, 'pau_s': 20997}
proposed: {'util': 0.1491274226080856, 'idle': 0.8508725773919144, 'lifespan': 516.8695652173913, 'run_s': 3401, 'pau_s': 19390}
saved figure


In [14]:
"""
Container utilization + idle-retention figure.
Single-column, black & white (ACM/Middleware style), no titles,
legend sitting just above the plots.
"""
import pandas as pd, numpy as np
import matplotlib
# matplotlib.use("Agg")   # uncomment for headless/batch (no plt.show window)
import matplotlib.pyplot as plt

BASELINE = "../results/exp_20260602_170520/docker_log.csv"
PROPOSED = "../results/exp_20260602_154435/docker_log.csv"

def metrics(p):
    df = pd.read_csv(p, header=None, names=["ts","container","image","status"])
    df["status"] = df["status"].str.strip().str.lower()
    df["t"] = (df["ts"] - df["ts"].min())/1000.0
    sec = df.groupby("status").size()
    run, pau, rem = sec.get("running",0), sec.get("paused",0), sec.get("removing",0)
    total = run+pau+rem
    g = df.groupby("container")
    lifespan = (g["t"].max()-g["t"].min()).mean()
    return {"util": run/total, "idle": (pau+rem)/total, "lifespan": lifespan}

b, p = metrics(BASELINE), metrics(PROPOSED)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.linewidth": 0.8,
    "hatch.linewidth": 0.6,
})

# extra top room for the legend, but no title gap
fig, axes = plt.subplots(1, 2, figsize=(3.4, 1.6))
systems = ["OpenWhisk", "NMIG"]
x = np.arange(2)
w = 0.6

# Panel (a)
active = np.array([b["util"], p["util"]])*100
idle   = np.array([b["idle"], p["idle"]])*100
ax = axes[0]
bar_active = ax.bar(x, active, w, label="Active", color="white",
                    edgecolor="black", hatch="////", linewidth=0.8)
bar_idle = ax.bar(x, idle, w, bottom=active, label="Idle", color="0.75",
                  edgecolor="black", linewidth=0.8)
for i in range(2):
    ax.text(i, active[i]+2, f"{active[i]:.1f}%", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel("Lifetime (%)")
ax.set_ylim(0, 105)

# Panel (b)
ax = axes[1]
life = [b["lifespan"], p["lifespan"]]
ax.bar(x, life, w, color="white", edgecolor="black", hatch="xxxx", linewidth=0.8)
for i, v in enumerate(life):
    ax.text(i, v+30, f"{v:.0f}s", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel("Mean lifespan (s)")
ax.set_ylim(0, max(life)*1.18)

for ax in axes:
    ax.tick_params(labelsize=7, length=3, width=0.6)
    for s in ["top","right"]:
        ax.spines[s].set_visible(False)

# single FIGURE-level legend, just above the axes (not floating high)
fig.legend([bar_active, bar_idle], ["Active", "Idle"],
           loc="upper center", ncol=2, frameon=False, fontsize=7,
           handlelength=1.2, columnspacing=1.0, bbox_to_anchor=(0.5, 1.02))

# leave a small strip at the top for the legend, no title gap
fig.tight_layout(pad=0.4, w_pad=1.0, rect=[0, 0, 1, 0.92])
fig.savefig("container_outputs/container_utilization.pdf", bbox_inches="tight")
fig.savefig("container_outputs/container_utilization.png", dpi=300, bbox_inches="tight")
print("saved")
plt.show()

saved


In [16]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [17]:
experiments = [
    {"id": "exp_20260602_170520", "name": "Openwhisk"},
    {"id": "exp_20260602_154435", "name": "NMIG"},

]

In [18]:
# import pandas as pd
# import json

def parse_json_safe(val):
    if pd.isna(val) or val == "None":
        return None
    if isinstance(val, dict):  # Already a dict
        return val
    try:
        return json.loads(val)
    except Exception:
        return None

def extract_name(res,name):
    if isinstance(res, dict):
        return res.get(name)
    return None




def extract_error_message(res):
    if isinstance(res, dict):
        resp = res.get("response", {})
        if not resp.get("success", True):
            return resp.get("result", {}).get("error")
    return None

def has_cuda_error(msg):
    if isinstance(msg, str):
        return "cuda" in msg.lower() or "cudnn" in msg.lower()
    return False

def extract_success(res):
    if isinstance(res, dict):
        return res.get("response", {}).get("success", True)
    return False



In [19]:
# plt.figure(figsize=(4, 3.5))
results = {}
df_res = {}

def pick_particular_column(rp,col="initTime"):
    # rp is expected to be a dict with key 'annotations' that is a list of dicts
    if not isinstance(rp, dict):
        return 0
    items = rp.get("annotations", []) or []
    for it in items:
        if isinstance(it, dict) and it.get("key") == col:
            return (it.get("value", 0))/1000
    return 0

    
for exp in experiments:
    # load the CSV for this experiment
    path = f"../results/{exp['id']}/results_updated.csv"
    df = pd.read_csv(path)

    # parse JSON and compute latency
    df["result_parsed"] = df["result"].apply(parse_json_safe)
    df["name"] = df["result_parsed"].apply(lambda res: extract_name(res, "name"))
    df['name'] = df['name'].str.replace(r'_p$', '', regex=True)
    df["start"] = df["result_parsed"].apply(lambda res: extract_name(res, "start"))
    df["end"] = df["result_parsed"].apply(lambda res: extract_name(res, "end"))
    df['latency'] = (df['end'] - df['start'])/1000
    df["error_message"] = df["result_parsed"].apply(extract_error_message)
    df["has_cuda_or_cudnn_error"] = df["error_message"].apply(has_cuda_error)
    df["success"] = df["result_parsed"].apply(extract_success)
    df["error"] = df["result_parsed"].isna() | (~df["success"])
    df["initTime"] = df["result_parsed"].apply(lambda x: pick_particular_column(x, col="initTime"))
    df["waitTime"] = df["result_parsed"].apply(lambda x: pick_particular_column(x, col="waitTime"))
    

    df_res[exp['name']] = df
    # error_counts = df['error'].value_counts()
    # results[exp['name']] = {}
    # results[exp['name']]['true_val'] = error_counts.get(True, 0)
    # results[exp['name']]['false_val'] = error_counts.get(False, 0)
    # results[exp['name']]['false_val'] = df['latency'].mean()
    # df_valid = df[df["error"] != True]
    # avg_latency_per_name_dict = df_valid.groupby("name")["latency"].mean().to_dict()
    # print(avg_latency_per_name_dict)
    # df_res[exp['
    print(exp)
    
   
    # df = df[df['latency'] < 300]

print(results)

{'id': 'exp_20260602_170520', 'name': 'Openwhisk'}
{'id': 'exp_20260602_154435', 'name': 'NMIG'}
{}


In [20]:
# for l in experiments:
# count_zero = (df_res['Openwhisk']["initTime"] == 0).sum()

for l in df_res:
    count_zero_initTime = (df_res[l]["initTime"] == 0).sum()
    print(f"{l} - {count_zero_initTime}")
print(20*"*")
for l in df_res:
    count_zero_initTime = (df_res[l]["waitTime"] == 0).sum()
    print(f"{l} - {count_zero_initTime}")

Openwhisk - 286
NMIG - 254
********************
Openwhisk - 0
NMIG - 0


In [23]:
df_res['NMIG']

,timestamp,func,activation_id,result,result_parsed,name,start,end,latency,error_message,has_cuda_or_cudnn_error,success,error,initTime,waitTime
0,1780382713000,762835950e81a11cd04cedcb05275dc111c651625d5750...,42de73b61adb471d9e73b61adb671d54,"{\n ""namespace"": ""guest"",\n ""name"": ""dis...","{'namespace': 'guest', 'name': 'distilgpt2_p',...",distilgpt2,1780382715847,1780382719941,4.094,None,False,True,False,0.169,1.991
1,1780382723000,313c03f53a0d31f70aec25f62efb33e7dd779725ca4af5...,a515e6bf809946db95e6bf8099c6db77,"{\n ""namespace"": ""guest"",\n ""name"": ""goo...","{'namespace': 'guest', 'name': 'googlenet_p', ...",googlenet,1780382725955,1780382727692,1.737,None,False,True,False,0.160,2.027
2,1780382725000,762835950e81a11cd04cedcb05275dc111c651625d5750...,3617cea1ce1d4bdc97cea1ce1dabdcd0,"{\n ""namespace"": ""guest"",\n ""name"": ""dis...","{'namespace': 'guest', 'name': 'distilgpt2_p',...",distilgpt2,1780382725975,1780382730007,4.032,None,False,True,False,0.000,0.005
3,1780382729000,313c03f53a0d31f70aec25f62efb33e7dd779725ca4af5...,7619eea953784c7099eea953781c70f0,"{\n ""namespace"": ""guest"",\n ""name"": ""goo...","{'namespace': 'guest', 'name': 'googlenet_p', ...",googlenet,1780382729013,1780382730519,1.506,None,False,True,False,0.000,0.005
4,1780382730000,762835950e81a11cd04cedcb05275dc111c651625d5750...,5b9fa82b143a484c9fa82b143a584cc8,"{\n ""namespace"": ""guest"",\n ""name"": ""dis...","{'namespace': 'guest', 'name': 'distilgpt2_p',...",distilgpt2,1780382730049,1780382733830,3.781,None,False,True,False,0.000,0.004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,1780386233000,556ccf8758c8c2a20082c161e955405e950439f0503522...,005a4e9826be44a39a4e9826be44a376,"{\n ""namespace"": ""guest"",\n ""name"": ""inc...","{'namespace': 'guest', 'name': 'inception_p', ...",inception,1780386233278,1780386235727,2.449,None,False,True,False,0.000,0.050
296,1780386240000,9b61fd55aa093a2d172db1a68a60af5cf6cbfa7f5ea1fb...,2ab420e941f54f7bb420e941f5af7b44,"{\n ""namespace"": ""guest"",\n ""name"": ""ber...","{'namespace': 'guest', 'name': 'bert_p', 'vers...",bert,1780386242236,1780386245131,2.895,None,False,True,False,0.155,1.960
297,1780386259000,313c03f53a0d31f70aec25f62efb33e7dd779725ca4af5...,0f8745de5b80488f8745de5b80488fc6,"{\n ""namespace"": ""guest"",\n ""name"": ""goo...","{'namespace': 'guest', 'name': 'googlenet_p', ...",googlenet,1780386259381,1780386260876,1.495,None,False,True,False,0.000,0.045
298,1780386266000,762835950e81a11cd04cedcb05275dc111c651625d5750...,0c69dd2f30754849a9dd2f30751849a1,"{\n ""namespace"": ""guest"",\n ""name"": ""dis...","{'namespace': 'guest', 'name': 'distilgpt2_p',...",distilgpt2,1780386266429,1780386270250,3.821,None,False,True,False,0.000,0.046
